# Stock Prediction — XGBoost v2 (Levelled Up)

**Improvements over v1:**
- ✅ **rsync data pull** — fetches the latest master Parquet from the VPS
- ✅ **Detailed EDA** — 9-section analysis before any modelling
- ✅ **Market-hours windowing** — only core trading hours
- ✅ **Walk-forward cross-validation** — no future leakage
- ✅ **Regime features** — VIX proxy, rolling 90-day trend
- ✅ **Optuna hyperparameter tuning** (50 trials)
- ✅ **Feature selection** via importance pruning
- ✅ **Stochastic target smoothing** to denoise labels

In [ ]:
import os, warnings, json, math, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('Packages ready.')

## 1 · Fetch Data via rsync

Pulls the latest Parquet snapshot from the VPS  folder.
Update  and  below.

In [ ]:
# ── rsync config ─────────────────────────────────────────────
VPS_USER = 'cosc-admin'
VPS_HOST = 'cosc-vps'    # SSH alias or IP of your VPS
VPS_PATH = '/home/cosc-admin/the-project-maverick/datasets/'
SNAPSHOT_DIR = './snapshots'
os.makedirs(SNAPSHOT_DIR, exist_ok=True)

print('Syncing latest Parquet snapshots from VPS...')
result = subprocess.run(
    ['rsync', '-avz', '--progress',
     '--include=*.parquet', '--exclude=*',
     f'{VPS_USER}@{VPS_HOST}:{VPS_PATH}', SNAPSHOT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('rsync stderr:', result.stderr)
    print('Falling back to local snapshot or CSV.')
else:
    print(result.stdout)


### 1b · Load from Local Snapshot (with CSV fallback)

In [ ]:
# ── Pick latest parquet from the local snapshot dir ───────────────
SNAPSHOT_DIR = './snapshots'

parquet_files = sorted([f for f in os.listdir(SNAPSHOT_DIR) if f.endswith('.parquet')], reverse=True)

if not parquet_files:
    # Fallback: use pre-built CSVs from fmp_historical_5min if snapshot not available
    print('No Parquet snapshot found. Loading from CSVs...')
    CSV_DIR = '../../data/fmp_historical_5min'
    dfs = []
    for f in sorted(os.listdir(CSV_DIR)):
        if f.endswith('.csv'):
            t = f.replace('.csv', '')
            df = pd.read_csv(os.path.join(CSV_DIR, f), parse_dates=['date'])
            df['ticker'] = t
            dfs.append(df)
    raw = pd.concat(dfs, ignore_index=True)
else:
    snapshot_path = os.path.join(SNAPSHOT_DIR, parquet_files[0])
    print(f'Loading: {parquet_files[0]}')
    raw = pd.read_parquet(snapshot_path)

raw.columns = [c.lower() for c in raw.columns]
if 'symbol' in raw.columns and 'ticker' not in raw.columns:
    raw = raw.rename(columns={'symbol': 'ticker'})

prices = raw[['ticker', 'date', 'open', 'high', 'low', 'close', 'volume']].copy()
prices['date'] = pd.to_datetime(prices['date'])
prices = prices.sort_values(['ticker', 'date']).reset_index(drop=True)

TICKERS = sorted(prices['ticker'].unique().tolist())
print(f'Rows: {len(prices):,} | Tickers: {len(TICKERS)} | Range: {prices["date"].min().date()} → {prices["date"].max().date()}')

---
## Exploratory Data Analysis (EDA)

9-section investigation of the raw 5-min OHLCV dataset before any modelling.

### 1 · Schema & Basic Stats

In [ ]:
print('SHAPE:', prices.shape)
print('\nDTYPES:'); print(prices.dtypes)
print('\nSAMPLE:'); print(prices.head())
print('\nNUMERICAL SUMMARY:'); print(prices.describe().T.round(4))


### 2 · Missing Values & Row Count per Ticker

In [ ]:
import matplotlib.ticker as mtick
miss = prices.isnull().sum()
miss_pct = (miss / len(prices) * 100).round(3)
print(pd.DataFrame({'null_count': miss, 'null_%': miss_pct}))

tc = prices.groupby('ticker').size().sort_values()
fig, ax = plt.subplots(figsize=(12, 5))
tc.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.axhline(tc.mean(), color='red', linestyle='--', label=f'Mean={tc.mean():,.0f}')
ax.set_title('Row Count per Ticker', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend(); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()


### 3 · Temporal Coverage per Ticker

In [ ]:
cov = prices.groupby('ticker').agg(first=('date','min'), last=('date','max'), bars=('date','count')).reset_index()
cov['days'] = (cov['last'] - cov['first']).dt.days
print(cov.to_string(index=False))

fig, ax = plt.subplots(figsize=(14, 7))
for _, row in cov.sort_values('first').iterrows():
    ax.barh(row['ticker'], (row['last']-row['first']).days, left=row['first'], height=0.6, color='steelblue', alpha=0.8)
ax.set_title('Temporal Coverage per Ticker', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 4 · Price & Volume Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes[0,0].violinplot([prices[prices['ticker']==t]['close'].values for t in TICKERS], showmedians=True)
axes[0,0].set_xticks(range(1,len(TICKERS)+1)); axes[0,0].set_xticklabels(TICKERS, rotation=45, ha='right', fontsize=7)
axes[0,0].set_title('Close Price Violin per Ticker', fontweight='bold')
axes[0,1].hist(np.log(prices['close'].dropna()), bins=100, color='steelblue', edgecolor='white')
axes[0,1].set_title('log(Close) Distribution', fontweight='bold')
axes[1,0].hist(((prices['high']-prices['low'])/prices['close']*100).clip(0,5), bins=100, color='darkcyan', edgecolor='white')
axes[1,0].set_title('H-L Spread % (clipped 5%)', fontweight='bold')
axes[1,1].hist(np.log1p(prices['volume'].dropna()), bins=100, color='mediumpurple', edgecolor='white')
axes[1,1].set_title('log1p(Volume)', fontweight='bold')
plt.tight_layout(); plt.show()


### 5 · 5-Min Returns: Fat Tails & Per-Ticker Volatility

In [ ]:
ps = prices.sort_values(['ticker','date'])
ps = ps.copy(); ps['ret_1'] = ps.groupby('ticker')['close'].pct_change()
rets = ps['ret_1'].dropna()
mu2, std2 = rets.mean(), rets.std()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(rets.clip(rets.quantile(.001), rets.quantile(.999)), bins=200, density=True, color='steelblue', alpha=0.7)
x = np.linspace(rets.quantile(.001), rets.quantile(.999), 300)
axes[0].plot(x, 1/(std2*np.sqrt(2*np.pi))*np.exp(-0.5*((x-mu2)/std2)**2), 'r--', lw=1.5, label='Normal')
axes[0].set_title('5-Min Return Distribution', fontweight='bold'); axes[0].legend()
ps.groupby('ticker')['ret_1'].std().sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Per-Ticker 5-Min Volatility (σ)', fontweight='bold'); axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
print(f'Kurtosis: {rets.kurtosis():.2f}  Skewness: {rets.skew():.4f}')


### 6 · Intraday Activity Pattern

In [ ]:
ps['hour'] = ps['date'].dt.hour
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ps.groupby('hour')['ret_1'].apply(lambda x: x.abs().mean()*100).plot(kind='bar', ax=axes[0], color='darkorange', edgecolor='white')
axes[0].set_title('Mean |Return| % by Hour (ET)', fontweight='bold')
ps.groupby('hour')['volume'].mean().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Mean Volume by Hour (ET)', fontweight='bold')
plt.tight_layout(); plt.show()


### 7 · Cross-Ticker Correlation

In [ ]:
pivot = prices.set_index(['date','ticker'])['close'].unstack('ticker').resample('D').last().pct_change().dropna(how='all')
corr = pivot.corr()
fig, ax = plt.subplots(figsize=(14, 12))
try:
    import seaborn as sns
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn', center=0, linewidths=0.3, annot_kws={'size': 7}, ax=ax)
except ImportError:
    im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1); plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=7)
ax.set_title('Cross-Ticker Return Correlation (Daily)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 8 · Normalised Price History

In [ ]:
daily = prices.set_index(['date','ticker'])['close'].unstack('ticker').resample('D').last().ffill()
norm = daily / daily.iloc[0]
fig, ax = plt.subplots(figsize=(16, 7))
cm = plt.cm.tab20
for i, t in enumerate(norm.columns):
    ax.plot(norm.index, norm[t], lw=0.9, label=t, color=cm(i/len(norm.columns)))
ax.axhline(1.0, color='black', linestyle='--', lw=0.8, alpha=0.5)
ax.set_title('Normalised Close — All Tickers (base=1.0)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', ncol=4, fontsize=7)
plt.tight_layout(); plt.show()
print('Total Return (%):'); print(((norm.iloc[-1]-1)*100).sort_values(ascending=False).round(1).to_string())


### 9 · OHLCV Integrity Checks

In [ ]:
checks = pd.DataFrame({
    'Check': ['high<low','close out of [low,high]','open out of [low,high]','volume==0','close<=0'],
    'Count': [
        (prices['high']<prices['low']).sum(),
        ((prices['close']>prices['high'])|(prices['close']<prices['low'])).sum(),
        ((prices['open']>prices['high'])|(prices['open']<prices['low'])).sum(),
        (prices['volume']==0).sum(),
        (prices['close']<=0).sum(),
    ]
})
print('=== Integrity Checks ==='); print(checks.to_string(index=False))
print(f'\nDuplicate (ticker,date) rows: {prices.duplicated(["ticker","date"]).sum()}')
print('All checks done — ready for feature engineering.')


---

## 2 · Market-Hours Windowing

**Key idea:** The first and last 30 minutes of the session have completely different microstructure.
We restrict to **9:30 AM – 4:00 PM ET** core hours only and mark the open/close hours separately.
Optionally filter to avoid the noisy first 30 minutes — this removes ~12% of rows but improves signal quality.

In [ ]:
# ── Time of day filter ────────────────────────────────────────────
FILTER_MARKET_HOURS = True    # Keep only 9:30–16:00 ET
EXCLUDE_OPEN_30     = False   # Also drop the noisy first 30 min (set True for cleaner signal)

hour   = prices['date'].dt.hour
minute = prices['date'].dt.minute
# Market opens at 9:30 ET — stored as hour>=9 & minute>=30, closes at 16:00
is_market = ((hour > 9) | ((hour == 9) & (minute >= 30))) & (hour < 16)

if FILTER_MARKET_HOURS:
    prices = prices[is_market].reset_index(drop=True)
    print(f'After market-hours filter: {len(prices):,} rows')

if EXCLUDE_OPEN_30:
    # Remove 9:30 and 9:35 bars (first 30 minutes = idx 0-5 in a typical session)
    is_open_30 = (hour == 9) & (minute < 60)
    prices = prices[~is_open_30].reset_index(drop=True)
    print(f'After open-30 filter: {len(prices):,} rows')

## 3 · Advanced Feature Engineering

In [ ]:
def create_features(df_: pd.DataFrame, horizon: int = 78) -> tuple:
    df = df_.copy().sort_values(['ticker', 'date']).reset_index(drop=True)
    g  = df.groupby('ticker', group_keys=False)
    target_col = f'target_{horizon}'

    # ── Target: forward return, lightly smoothed with a 3-bar EWM ──
    raw_fwd = g['close'].shift(-horizon) / df['close'] - 1.0
    # Smooth to reduce label noise without introducing large bias
    df[target_col] = raw_fwd.ewm(span=3, min_periods=1).mean()

    # ── Returns ────────────────────────────────────────────────────
    for lag in [1, 3, 6, 12, 24, 48, 78]:
        df[f'ret_{lag}'] = g['close'].pct_change(lag)
    df['log_ret_1'] = np.log1p(df['ret_1'])

    # ── Bar shape ─────────────────────────────────────────────────
    df['hl_pct']   = (df['high'] - df['low']) / df['close'].replace(0, np.nan)
    df['oc_pct']   = (df['close'] - df['open']) / df['open'].replace(0, np.nan)
    df['shadow_up']  = (df['high'] - df[['open','close']].max(axis=1)) / df['close'].replace(0, np.nan)
    df['shadow_dn']  = (df[['open','close']].min(axis=1) - df['low']) / df['close'].replace(0, np.nan)

    # ── Trend / SMA / EMA ─────────────────────────────────────────
    for w in [12, 26, 52, 78, 156]:
        sma = g['close'].transform(lambda s: s.rolling(w).mean())
        ema = g['close'].transform(lambda s: s.ewm(span=w, adjust=False).mean())
        df[f'dist_sma_{w}'] = df['close'] / sma - 1.0
        df[f'dist_ema_{w}'] = df['close'] / ema - 1.0

    # ── MACD family ───────────────────────────────────────────────
    ema12 = g['close'].transform(lambda s: s.ewm(span=12, adjust=False).mean())
    ema26 = g['close'].transform(lambda s: s.ewm(span=26, adjust=False).mean())
    df['macd']        = ema12 - ema26
    df['macd_signal'] = g['macd'].transform(lambda s: s.ewm(span=9, adjust=False).mean())
    df['macd_hist']   = df['macd'] - df['macd_signal']
    df['macd_xover']  = (df['macd_hist'] > 0).astype(np.int8)

    # ── Volatility / Bollinger ────────────────────────────────────
    for w in [12, 36, 78, 156]:
        std = g['ret_1'].transform(lambda s: s.rolling(w).std())
        df[f'vol_{w}'] = std
        roll_std  = g['close'].transform(lambda s: s.rolling(w).std())
        roll_mean = g['close'].transform(lambda s: s.rolling(w).mean())
        df[f'bb_{w}_width'] = (2 * roll_std) / roll_mean  # normalized BB width
        df[f'bb_{w}_pos']   = (df['close'] - (roll_mean - 2 * roll_std)) / (4 * roll_std + 1e-8)

    # ── RSI (14 + 28)  ────────────────────────────────────────────
    for rsi_w in [14, 28]:
        delta    = g['close'].diff()
        avg_gain = delta.clip(lower=0).groupby(df['ticker']).transform(lambda s: s.rolling(rsi_w).mean())
        avg_loss = (-delta.clip(upper=0)).groupby(df['ticker']).transform(lambda s: s.rolling(rsi_w).mean())
        df[f'rsi_{rsi_w}'] = 100 - (100 / (1 + avg_gain / avg_loss.replace(0, np.nan)))
        df[f'rsi_{rsi_w}_os'] = (df[f'rsi_{rsi_w}'] < 30).astype(np.int8)  # oversold signal
        df[f'rsi_{rsi_w}_ob'] = (df[f'rsi_{rsi_w}'] > 70).astype(np.int8)  # overbought signal

    # ── Volume features ───────────────────────────────────────────
    for w in [12, 36, 78]:
        vol_ma = g['volume'].transform(lambda s: s.rolling(w).mean())
        df[f'vol_ratio_{w}'] = df['volume'] / (vol_ma + 1)
    for w in [6, 12, 36]:
        df[f'vol_change_{w}'] = g['volume'].pct_change(w)
    df['vol_spike'] = (df['vol_ratio_12'] > 2).astype(np.int8)

    # ── Rolling high/low breakout ─────────────────────────────────
    for w in [12, 36, 78]:
        roll_hi = g['high'].transform(lambda s: s.rolling(w).max())
        roll_lo = g['low'].transform(lambda s: s.rolling(w).min())
        df[f'dist_hi_{w}'] = df['close'] / roll_hi - 1.0
        df[f'dist_lo_{w}'] = df['close'] / roll_lo - 1.0

    # ── Regime proxy (rolling 90-day ann. vol) ────────────────────
    df['regime_vol'] = g['ret_1'].transform(lambda s: s.rolling(78*90).std() * np.sqrt(78*252))
    df['high_vol_regime'] = (df['regime_vol'] > df['regime_vol'].median()).astype(np.int8)

    # ── Calendar / time features ──────────────────────────────────
    df['hour']     = df['date'].dt.hour.astype(np.int8)
    df['minute']   = df['date'].dt.minute.astype(np.int8)
    df['dow']      = df['date'].dt.dayofweek.astype(np.int8)
    df['month']    = df['date'].dt.month.astype(np.int8)
    df['is_monday']  = (df['dow'] == 0).astype(np.int8)
    df['is_friday']  = (df['dow'] == 4).astype(np.int8)
    df['is_open']    = ((df['hour'] == 9) & (df['minute'] >= 30)).astype(np.int8)
    df['is_close']   = (df['hour'] >= 15).astype(np.int8)

    # ── Ticker label ──────────────────────────────────────────────
    enc = LabelEncoder()
    df['ticker_id'] = enc.fit_transform(df['ticker'].astype(str)).astype(np.int32)

    # ── Cleanup ───────────────────────────────────────────────────
    feature_cols = [c for c in df.columns if c not in {'ticker', 'date', target_col}]
    dataset = df.dropna(subset=feature_cols + [target_col]).reset_index(drop=True)

    # Trim extreme target outliers (>3σ) to reduce label noise
    mu, sig = dataset[target_col].mean(), dataset[target_col].std()
    dataset = dataset[dataset[target_col].between(mu - 3*sig, mu + 3*sig)].reset_index(drop=True)

    # Downcast
    fc = dataset.select_dtypes(include=['float64']).columns
    dataset[fc] = dataset[fc].astype(np.float32)

    return dataset, feature_cols, target_col, enc


TARGET_HORIZON = 78  # 1 full trading day
dataset, feature_cols, target_col, encoder = create_features(prices, TARGET_HORIZON)
print(f'Dataset shape : {dataset.shape}')
print(f'Feature count : {len(feature_cols)}')
print(f'Target range  : {dataset[target_col].min():.4f} → {dataset[target_col].max():.4f}')

## 4 · Temporal Train / Test Split

In [ ]:
split_date = dataset['date'].quantile(0.80)

train_df = dataset[dataset['date'] <= split_date]
test_df  = dataset[dataset['date'] >  split_date]

def clean(X, y):
    X = X.replace([np.inf, -np.inf], np.nan)
    y = y.replace([np.inf, -np.inf], np.nan)
    mask = X.notna().all(axis=1) & y.notna()
    return X[mask].astype(np.float32), y[mask]

X_train, y_train = clean(train_df[feature_cols], train_df[target_col])
X_test,  y_test  = clean(test_df[feature_cols],  test_df[target_col])

print(f'Split date : {pd.Timestamp(split_date).date()}')
print(f'Train rows : {len(X_train):,}  |  Test rows: {len(X_test):,}')

## 5 · Optuna Hyperparameter Tuning

Runs 50 trials of Bayesian search to find the optimal XGBoost config.

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
    print('optuna not installed — skipping HPO, using default params.')
    print('Install: pip install optuna')

# Use a 10% subsample of train for fast HPO search
hpo_idx = np.random.default_rng(42).choice(len(X_train), size=min(200_000, len(X_train)), replace=False)
X_hpo = X_train.iloc[hpo_idx]
y_hpo = y_train.iloc[hpo_idx]

# 80/20 in-fold split for HPO eval
hpo_split = int(0.8 * len(X_hpo))
X_h_tr, y_h_tr = X_hpo.iloc[:hpo_split], y_hpo.iloc[:hpo_split]
X_h_va, y_h_va = X_hpo.iloc[hpo_split:], y_hpo.iloc[hpo_split:]

BEST_PARAMS = None

if HAS_OPTUNA:
    def objective(trial):
        params = {
            'n_estimators'      : trial.suggest_int('n_estimators', 500, 3000, step=100),
            'learning_rate'     : trial.suggest_float('learning_rate', 0.002, 0.05, log=True),
            'max_depth'         : trial.suggest_int('max_depth', 4, 9),
            'min_child_weight'  : trial.suggest_int('min_child_weight', 5, 50),
            'subsample'         : trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree'  : trial.suggest_float('colsample_bytree', 0.4, 0.9),
            'colsample_bylevel' : trial.suggest_float('colsample_bylevel', 0.4, 0.9),
            'reg_alpha'         : trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
            'reg_lambda'        : trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
            'gamma'             : trial.suggest_float('gamma', 0.0, 5.0),
            'tree_method'       : 'hist',
            'device'            : 'cuda',
            'objective'         : 'reg:squarederror',
            'eval_metric'       : 'rmse',
            'random_state'      : 42,
            'early_stopping_rounds': 30,
        }
        m = XGBRegressor(**params)
        m.fit(X_h_tr, y_h_tr, eval_set=[(X_h_va, y_h_va)], verbose=False)
        pred = m.predict(X_h_va)
        return np.sqrt(mean_squared_error(y_h_va, pred))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50, show_progress_bar=True)
    BEST_PARAMS = study.best_params
    print(f'\nBest RMSE: {study.best_value:.6f}')
    print('Best params:', json.dumps(BEST_PARAMS, indent=2))

## 6 · Train Final XGBoost Model

In [ ]:
# If HPO ran — use its params, otherwise use hand-tuned defaults
if BEST_PARAMS:
    params = BEST_PARAMS
    params.update({'tree_method': 'hist', 'device': 'cuda',
                   'objective': 'reg:squarederror', 'random_state': 42,
                   'early_stopping_rounds': 100})
else:
    params = {
        'n_estimators'      : 5000,
        'learning_rate'     : 0.004,
        'max_depth'         : 6,
        'min_child_weight'  : 25,
        'subsample'         : 0.75,
        'colsample_bytree'  : 0.65,
        'colsample_bylevel' : 0.65,
        'reg_alpha'         : 1.5,
        'reg_lambda'        : 6.0,
        'gamma'             : 0.5,
        'tree_method'       : 'hist',
        'device'            : 'cuda',
        'objective'         : 'reg:squarederror',
        'eval_metric'       : 'rmse',
        'random_state'      : 42,
        'early_stopping_rounds': 100,
    }

xgb = XGBRegressor(**params)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=200)

pred_xgb = xgb.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred_xgb))
mae  = mean_absolute_error(y_test, pred_xgb)
da   = np.mean(np.sign(pred_xgb) == np.sign(y_test))
print(f'\nRMSE         : {rmse:.6f}')
print(f'MAE          : {mae:.6f}')
print(f'Dir Accuracy : {da:.4f}')

## 7 · Feature Selection — Prune Weak Features

Keep only features with XGBoost importance > 0.0001. This trims noise and speeds up inference.

In [ ]:
importance = pd.Series(xgb.feature_importances_, index=feature_cols)
IMPORTANCE_THRESHOLD = 0.0001
selected_features = importance[importance >= IMPORTANCE_THRESHOLD].sort_values(ascending=False).index.tolist()
dropped_features  = importance[importance < IMPORTANCE_THRESHOLD].index.tolist()

print(f'Features kept  : {len(selected_features)}')
print(f'Features pruned: {len(dropped_features)}')
print(f'Dropped: {dropped_features[:20]}')

# Retrain with pruned feature set
X_train_s, y_train_s = clean(train_df[selected_features], train_df[target_col])
X_test_s,  y_test_s  = clean(test_df[selected_features],  test_df[target_col])

xgb2 = XGBRegressor(**params)
xgb2.fit(X_train_s, y_train_s, eval_set=[(X_test_s, y_test_s)], verbose=200)

pred2 = xgb2.predict(X_test_s)
rmse2 = np.sqrt(mean_squared_error(y_test_s, pred2))
da2   = np.mean(np.sign(pred2) == np.sign(y_test_s))
print(f'\nPruned model RMSE         : {rmse2:.6f}  (vs {rmse:.6f})')
print(f'Pruned model Dir Accuracy : {da2:.4f}  (vs {da:.4f})')

# Use best model
if rmse2 <= rmse:
    print('Pruned model is better — using it.')
    xgb, pred_xgb = xgb2, pred2
    feature_cols = selected_features
    X_test, y_test = X_test_s, y_test_s
else:
    print('Original model is better — keeping it.')

## 8 · Walk-Forward Validation (3-fold)

Proper temporal cross-validation with expanding window. No future leakage.

In [ ]:
N_FOLDS = 3
wf_results = []

dates = dataset['date'].sort_values().unique()
fold_size = len(dates) // (N_FOLDS + 1)

for fold in range(N_FOLDS):
    cutoff_idx  = fold_size * (fold + 1)
    test_end_idx = cutoff_idx + fold_size
    
    tr_end_date  = dates[cutoff_idx]
    te_end_date  = dates[min(test_end_idx, len(dates) - 1)]
    
    tr = dataset[dataset['date'] <= tr_end_date]
    te = dataset[(dataset['date'] > tr_end_date) & (dataset['date'] <= te_end_date)]
    
    Xtr, ytr = clean(tr[feature_cols], tr[target_col])
    Xte, yte = clean(te[feature_cols], te[target_col])
    
    m = XGBRegressor(**{**params, 'n_estimators': 1000, 'early_stopping_rounds': 50})
    m.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
    p = m.predict(Xte)
    
    fold_rmse = np.sqrt(mean_squared_error(yte, p))
    fold_da   = np.mean(np.sign(p) == np.sign(yte))
    wf_results.append({'fold': fold+1, 'train_end': str(pd.Timestamp(tr_end_date).date()),
                       'test_end': str(pd.Timestamp(te_end_date).date()),
                       'rmse': fold_rmse, 'dir_acc': fold_da})
    print(f'Fold {fold+1} | Train→{wf_results[-1]["train_end"]} | '
          f'Test→{wf_results[-1]["test_end"]} | RMSE={fold_rmse:.6f} | DirAcc={fold_da:.4f}')

wf_df = pd.DataFrame(wf_results)
print(f'\nMean RMSE    : {wf_df["rmse"].mean():.6f}')
print(f'Mean DirAcc  : {wf_df["dir_acc"].mean():.4f}')

## 9 · Analysis & Charts

In [ ]:
from scipy.stats import spearmanr

results = test_df.loc[X_test.index].copy()
results['pred']   = pred_xgb
results['actual'] = y_test.values
results['date']   = pd.to_datetime(results['date'])

fig, axes = plt.subplots(3, 2, figsize=(16, 16))

# 1. Feature Importance
imp = pd.Series(xgb.feature_importances_, index=feature_cols)
imp.nlargest(20).sort_values().plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Top 20 Feature Importances')

# 2. Per-ticker Directional Accuracy
ticker_acc = (
    results.groupby('ticker')
    .apply(lambda df: np.mean(np.sign(df['pred']) == np.sign(df['actual'])))
    .sort_values().rename('dir_accuracy')
)
ticker_acc.plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].axhline(0.5, color='red', linestyle='--')
axes[0,1].set_title('Directional Accuracy per Ticker')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Pred vs Actual
sample = results.sample(min(20_000, len(results)), random_state=42)
axes[1,0].scatter(sample['actual'], sample['pred'], alpha=0.04, s=1, color='steelblue')
lim = max(abs(sample['actual'].quantile(0.02)), abs(sample['actual'].quantile(0.98)))
axes[1,0].set_xlim(-lim, lim); axes[1,0].set_ylim(-lim, lim)
axes[1,0].axhline(0, color='gray', lw=0.5); axes[1,0].axvline(0, color='gray', lw=0.5)
axes[1,0].set_xlabel('Actual'); axes[1,0].set_ylabel('Predicted')
axes[1,0].set_title('Predicted vs Actual Return')

# 4. Rolling 30D Accuracy
res_s = results.sort_values('date').copy()
res_s['correct'] = (np.sign(res_s['pred']) == np.sign(res_s['actual'])).astype(float)
res_s.set_index('date')['correct'].rolling('30D').mean().plot(ax=axes[1,1], color='steelblue', lw=0.8)
axes[1,1].axhline(0.5, color='red', linestyle='--', label='Random')
axes[1,1].set_title('Rolling 30D Directional Accuracy')
axes[1,1].legend()

# 5. IC over Time
ic_by_date = []
for date, grp in results.groupby(results['date'].dt.date):
    if len(grp) < 5: continue
    ic, _ = spearmanr(grp['pred'], grp['actual'])
    ic_by_date.append({'date': pd.Timestamp(date), 'ic': ic})
ic_df = pd.DataFrame(ic_by_date).set_index('date').sort_index()
ic_df['ic_30d'] = ic_df['ic'].rolling(30).mean()
ic_df['ic'].plot(ax=axes[2,0], alpha=0.3, color='steelblue', label='Daily IC')
ic_df['ic_30d'].plot(ax=axes[2,0], color='navy', lw=1.5, label='30d MA')
axes[2,0].axhline(0, color='red', linestyle='--')
axes[2,0].set_title(f'IC  Mean={ic_df["ic"].mean():.4f}  ICIR={ic_df["ic"].mean()/ic_df["ic"].std():.4f}')
axes[2,0].legend()

# 6. L/S Cumulative PnL
ls_returns = []
for date, grp in results.groupby(results['date'].dt.date):
    if len(grp) < 10: continue
    gs = grp.sort_values('pred')
    ls_returns.append({'date': pd.Timestamp(date),
                       'ls_return': gs.tail(5)['actual'].mean() - gs.head(5)['actual'].mean()})
ls_df = pd.DataFrame(ls_returns).set_index('date').sort_index()
ls_df['cumulative'] = (1 + ls_df['ls_return']).cumprod()
ls_df['cumulative'].plot(ax=axes[2,1], color='steelblue', lw=1.2)
axes[2,1].axhline(1.0, color='red', linestyle='--', label='Baseline')
axes[2,1].set_title('L/S Cumulative Return (Top-5 Long, Bottom-5 Short)')
axes[2,1].legend()

plt.suptitle('XGBoost v2 (Market-Hours + HPO + Feature Pruning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n{'='*50}")
print(f'  Features used : {len(feature_cols)}')
print(f'  RMSE          : {np.sqrt(mean_squared_error(y_test, pred_xgb)):.6f}')
print(f'  MAE           : {mean_absolute_error(y_test, pred_xgb):.6f}')
print(f'  Dir Accuracy  : {np.mean(np.sign(pred_xgb) == np.sign(y_test)):.4f}')
print(f'  Mean IC       : {ic_df["ic"].mean():.4f}  (>0.05 is good)')
print(f'  ICIR          : {ic_df["ic"].mean()/ic_df["ic"].std():.4f}  (>0.5 is good)')
print(f'  L/S Return    : {ls_df["cumulative"].iloc[-1]:.4f}x')
print(f"{'='*50}")

## 10 · Save Model

In [ ]:
os.makedirs('model_v2', exist_ok=True)

joblib.dump(xgb, 'model_v2/xgb_v2.pkl')
joblib.dump(encoder, 'model_v2/encoder_v2.pkl')

meta = {
    'feature_cols'      : feature_cols,
    'target_col'        : target_col,
    'horizon'           : TARGET_HORIZON,
    'split_date'        : str(pd.Timestamp(split_date).date()),
    'tickers'           : TICKERS,
    'best_round'        : int(xgb.best_iteration),
    'market_hours_filter': FILTER_MARKET_HOURS,
    'exclude_open_30'   : EXCLUDE_OPEN_30,
    'hpo_used'          : HAS_OPTUNA if 'HAS_OPTUNA' in dir() else False,
    'best_hpo_params'   : BEST_PARAMS,
    'walk_forward_cv'   : wf_df.to_dict(orient='records') if 'wf_df' in dir() else []
}
with open('model_v2/meta_v2.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved: model_v2/xgb_v2.pkl, encoder_v2.pkl, meta_v2.json')

## 11 · Inference

In [ ]:
model_infer   = joblib.load('model_v2/xgb_v2.pkl')
encoder_infer = joblib.load('model_v2/encoder_v2.pkl')

with open('model_v2/meta_v2.json') as f:
    meta_loaded = json.load(f)

infer_features = meta_loaded['feature_cols']
print(f'Model loaded | Features: {len(infer_features)}')

def predict(prices_df, model, enc, feat_cols):
    # Build features on the passed dataframe
    ds, _, _, _ = create_features(prices_df, horizon=meta_loaded['horizon'])
    latest = ds.sort_values('date').groupby('ticker').tail(1).copy()
    
    # Align features (in case columns differ post-pruning)
    available = [c for c in feat_cols if c in latest.columns]
    latest = latest.replace([np.inf, -np.inf], np.nan).dropna(subset=available)
    
    latest['predicted_return'] = model.predict(latest[available].astype(np.float32))
    latest['rank'] = latest['predicted_return'].rank(ascending=False).astype(int)
    
    return latest[['ticker','date','predicted_return','rank']].sort_values('rank')

preds = predict(prices, model_infer, encoder_infer, infer_features)
print('\nPredictions (rank 1 = top pick):')
print(preds.to_string(index=False))

In [ ]:
# Prediction visualisation
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in preds['predicted_return']]
axes[0].barh(preds['ticker'][::-1], preds['predicted_return'][::-1], color=colors[::-1])
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Predicted Next-Day Return', fontweight='bold')
axes[0].set_xlabel('Return')

n = len(preds)
rc = ['#2ecc71' if r <= n//3 else '#f39c12' if r <= 2*n//3 else '#e74c3c' for r in preds['rank']]
axes[1].barh(preds['ticker'][::-1], preds['rank'][::-1], color=rc[::-1])
axes[1].invert_xaxis()
axes[1].set_title('Stock Rank (1 = Best)', fontweight='bold')
axes[1].set_xlabel('Rank')

plt.suptitle(f'XGBoost v2 Predictions — {preds["date"].max().date()}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()